# Phase 5 — Modeling on the Validation Folds

**Purpose.** Compare three model families on three feature steps, using only the 11 validation folds built in SQL (`cv_fold_rows`), and answer the project question with numbers: **what does the mortgage rate add once a county's own price momentum is known?** This notebook never reads the Embargo or Test periods. The Test period is scored once, later, with frozen models.

## Design, fixed before any result

| Item | Choice |
|---|---|
| Data | `cv_fold_rows`: Train-period rows only, 11 expanding-window folds, validation years 2010 to 2020 |
| Families | Logistic regression (inputs standardized inside each fold), random forest, histogram gradient boosting |
| Settings | Fixed, no tuning. Same settings in every fold and every step |
| Feature steps | **Baseline:** `momentum_1m`, `momentum_3m`, `momentum_6m`. **Step 2:** baseline + `rate_level`. **Step 3:** step 2 + `rate_roc_12m` + `rate_momentum_interaction` |
| Metrics | ROC-AUC (main) and PR-AUC, per fold, next to each fold's own base rate |
| Baseline lock | The baseline scores are saved to `outputs/phase5_baseline_lock.json` **before** steps 2 and 3 are fitted |
| Uncertainty | Bootstrap that resamples whole **quarters** (the 44 validation quarters), 2,000 times |
| A step "helps" | Its interval excludes zero **and** the gain is at least +0.01 ROC-AUC |
| A step is "robust" | It helps in at least 2 of the 3 families |
| Recommended model | The family and step with the highest average ROC-AUC across the 11 folds, with PR-AUC shown next to it |

**What this notebook does not do.** It does not score the Test period, does not choose a decision threshold, and does not refit models on all of Train. Those belong to Phase 6.

**Run time.** The random forest is the slow part: 33 fits in total. Progress prints after every family and step.

## 1. Setup and connection

In [1]:
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sqlalchemy import create_engine, text

In [2]:
load_dotenv(override=True)

DB_USER = os.environ.get("PGUSER", "postgres")
DB_PASSWORD = os.environ.get("PGPASSWORD", "")
DB_HOST = os.environ.get("PGHOST", "localhost")
DB_PORT = os.environ.get("PGPORT", "5432")
DB_NAME = os.environ.get("PGDATABASE", "housing_risk_sql")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

os.makedirs("../outputs", exist_ok=True)
with engine.connect() as conn:
    print(conn.execute(text("SELECT current_database();")).scalar())
    for name in ("cv_fold_rows", "quarterly_modeling_table"):
        assert conn.execute(text("SELECT to_regclass(:n)"), {"n": name}).scalar(), (
            f"STOP: {name} not found. Run notebook 04 first.")
print("cv_fold_rows and quarterly_modeling_table found.")

capstone_sql_variant
cv_fold_rows and quarterly_modeling_table found.


## 2. Load the fold rows and check them

`cv_fold_rows` lists, for each fold, its training rows and its validation rows, assigned by label quarter. Only Train-period rows can be in it. Before any model is fitted the notebook checks that the folds are what the SQL promised, because every later number rests on them.

In [3]:
fold_rows = pd.read_sql(
    "SELECT * FROM cv_fold_rows ORDER BY fold_id, fold_role, county_fips, quarter", engine)
for c in ("quarter", "label_quarter_end", "feature_quarter_end"):
    fold_rows[c] = pd.to_datetime(fold_rows[c])

ALL_FEATURES = ["momentum_1m", "momentum_3m", "momentum_6m",
                "rate_level", "rate_roc_12m", "rate_momentum_interaction"]

val = (fold_rows[fold_rows["fold_role"] == "val"]
       .sort_values(["fold_id", "county_fips", "quarter"]).reset_index(drop=True))
fold_ids = sorted(fold_rows["fold_id"].unique())

n_missing = int(fold_rows[ALL_FEATURES].isna().sum().sum())
only_train = set(fold_rows["split"].unique()) == {"train"}
min_val_pos = int(val.groupby("fold_id")["decline_label"].sum().min())
order_ok = all(
    fold_rows[(fold_rows["fold_id"] == f) & (fold_rows["fold_role"] == "train")]["label_quarter_end"].max()
    < val[val["fold_id"] == f]["label_quarter_end"].min()
    for f in fold_ids)
folds_per_quarter = int(val.groupby("quarter")["fold_id"].nunique().max())
n_val_quarters = int(val["quarter"].nunique())

print(f"Folds: {len(fold_ids)} | validation quarters: {n_val_quarters} | "
      f"missing feature values: {n_missing} | lowest validation fold: {min_val_pos} declines")
print(f"Only Train rows: {only_train} | each training window ends before its validation window: {order_ok} | "
      f"max folds sharing a quarter: {folds_per_quarter}")
assert len(fold_ids) == 11, "Expected 11 folds."
assert only_train, "A fold row comes from outside Train."
assert n_missing == 0, "A feature value is missing."
assert min_val_pos >= 20, "A validation fold has fewer than 20 declines."
assert order_ok, "A training window reaches into its validation window."
assert folds_per_quarter == 1 and n_val_quarters == 44, "Validation windows are not 11 separate years."

pooled_rows, pooled_pos = len(val), int(val["decline_label"].sum())
print(f"Pooled validation: {pooled_rows:,} rows, {pooled_pos:,} declines, {100 * pooled_pos / pooled_rows:.2f}%")
if (pooled_rows, pooled_pos) != (112734, 4252):
    print("NOTE: differs from the figures verified in notebook 04 (112,734 rows, 4,252 declines).")

fold_overview = pd.DataFrame({
    "train_rows": fold_rows[fold_rows["fold_role"] == "train"].groupby("fold_id").size(),
    "train_declines": fold_rows[fold_rows["fold_role"] == "train"].groupby("fold_id")["decline_label"].sum(),
    "val_rows": val.groupby("fold_id").size(),
    "val_declines": val.groupby("fold_id")["decline_label"].sum(),
})
fold_overview["val_rate_pct"] = (100 * fold_overview["val_declines"] / fold_overview["val_rows"]).round(2)
fold_overview

Folds: 11 | validation quarters: 44 | missing feature values: 0 | lowest validation fold: 71 declines
Only Train rows: True | each training window ends before its validation window: True | max folds sharing a quarter: 1
Pooled validation: 112,734 rows, 4,252 declines, 3.77%


,train_rows,train_declines,val_rows,val_declines,val_rate_pct
fold_id,,,,,
0,42722,3526,7892,882,11.18
1,50614,4408,8232,1121,13.62
2,58846,5529,8683,288,3.32
3,67529,5817,9667,301,3.11
4,77196,6118,9874,325,3.29
5,87070,6443,10049,163,1.62
6,97119,6606,10590,341,3.22
7,107709,6947,11886,312,2.62
8,119595,7259,11915,245,2.06


## 3. Feature steps, models and helpers

The settings are fixed here and never changed after a result is seen. Logistic regression sits in a pipeline, so its standardization is fitted on each fold's training rows only and never sees validation rows.

In [4]:
FEATURE_STEPS = {
    "baseline":          ["momentum_1m", "momentum_3m", "momentum_6m"],
    "step2_rate_level":  ["momentum_1m", "momentum_3m", "momentum_6m", "rate_level"],
    "step3_rate_change": ALL_FEATURES,
}
FAMILIES = ["logreg", "random_forest", "hist_gb"]
RANDOM_STATE = 42
N_BOOT = 2000
MIN_GAIN = 0.01


def make_model(family):
    if family == "logreg":
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    if family == "random_forest":
        return RandomForestClassifier(n_estimators=200, min_samples_leaf=5, max_depth=12,
                                      random_state=RANDOM_STATE, n_jobs=-1)
    if family == "hist_gb":
        return HistGradientBoostingClassifier(max_iter=150, random_state=RANDOM_STATE)
    raise ValueError(family)

In [5]:
train_by_fold = {f: fold_rows[(fold_rows["fold_id"] == f) & (fold_rows["fold_role"] == "train")]
                 for f in fold_ids}
val_pos = {f: np.where(val["fold_id"].to_numpy() == f)[0] for f in fold_ids}
val_y = val["decline_label"].to_numpy()


def run_step(step):
    # Fit every family on every fold for one feature step, return validation probabilities.
    feats = FEATURE_STEPS[step]
    out = {}
    for family in FAMILIES:
        proba = np.full(len(val), np.nan)
        t0 = time.time()
        for f in fold_ids:
            tr = train_by_fold[f]
            model = make_model(family).fit(tr[feats].to_numpy(), tr["decline_label"].to_numpy())
            proba[val_pos[f]] = model.predict_proba(val.loc[val_pos[f], feats].to_numpy())[:, 1]
        assert not np.isnan(proba).any()
        out[(family, step)] = proba
        print(f"{step:18s} {family:14s} 11 folds in {time.time() - t0:6.1f}s")
    return out


def score_folds(preds):
    rows = []
    for (family, step), p in preds.items():
        for f in fold_ids:
            y, s = val_y[val_pos[f]], p[val_pos[f]]
            rows.append({"family": family, "step": step, "fold_id": f, "n_val": len(y),
                         "n_positives": int(y.sum()), "base_rate": y.mean(),
                         "roc_auc": roc_auc_score(y, s), "pr_auc": average_precision_score(y, s)})
    return pd.DataFrame(rows)

## 4. Step 1: the baseline, locked first

The baseline (momentum only) is fitted and scored first. Its numbers are written to `outputs/phase5_baseline_lock.json` before steps 2 and 3 exist, so the reference point cannot move after the rate features are seen. If the file already exists, a re-run must reproduce it (within 1e-6) or the notebook stops.

In [6]:
preds_base = run_step("baseline")
scores_base = score_folds(preds_base)

base_summary = scores_base.groupby("family").agg(
    mean_roc_auc=("roc_auc", "mean"), std_roc_auc=("roc_auc", "std"),
    mean_pr_auc=("pr_auc", "mean"), std_pr_auc=("pr_auc", "std")).loc[FAMILIES]
base_summary.round(4)

baseline           logreg         11 folds in    1.8s
baseline           random_forest  11 folds in  113.1s
baseline           hist_gb        11 folds in    7.4s


,mean_roc_auc,std_roc_auc,mean_pr_auc,std_pr_auc
family,,,,
logreg,0.8364,0.0630,0.3290,0.1399
random_forest,0.8680,0.0351,0.3120,0.1305
hist_gb,0.8666,0.0359,0.3139,0.1342


In [7]:
LOCK_PATH = Path("../outputs/phase5_baseline_lock.json")
lock = {
    "purpose": "Baseline (momentum only) validation scores, saved before steps 2 and 3 were fitted.",
    "feature_columns": FEATURE_STEPS["baseline"],
    "n_folds": len(fold_ids),
    "validation_rows": int(len(val)),
    "validation_declines": int(val_y.sum()),
    "std_is_sample_std_across_folds": True,
    "random_state": RANDOM_STATE,
    "settings": {"logreg": "StandardScaler + LogisticRegression(max_iter=2000)",
                 "random_forest": "n_estimators=200, min_samples_leaf=5, max_depth=12",
                 "hist_gb": "max_iter=150"},
    "families": {fam: {k: float(v) for k, v in base_summary.loc[fam].items()} for fam in FAMILIES},
}
if LOCK_PATH.exists():
    old = json.loads(LOCK_PATH.read_text(encoding="utf-8"))
    gaps = [abs(old["families"][fam][k] - lock["families"][fam][k])
            for fam in FAMILIES for k in lock["families"][fam]]
    assert max(gaps) < 1e-6, ("The baseline no longer matches the locked file. Delete "
                              f"{LOCK_PATH} only if the data changed on purpose.")
    print(f"Baseline reproduces the locked file (largest gap {max(gaps):.1e}).")
else:
    LOCK_PATH.write_text(json.dumps(lock, indent=2), encoding="utf-8")
    print(f"Baseline locked in {LOCK_PATH}")

Baseline locked in ..\outputs\phase5_baseline_lock.json


## 5. Steps 2 and 3: adding the rate

In [8]:
preds_step2 = run_step("step2_rate_level")
preds_step3 = run_step("step3_rate_change")
all_preds = {**preds_base, **preds_step2, **preds_step3}
fold_scores = pd.concat([scores_base, score_folds(preds_step2), score_folds(preds_step3)],
                        ignore_index=True)
print(f"{len(all_preds)} models, {len(fold_scores)} fold scores")
assert len(all_preds) == 9 and len(fold_scores) == 99

step2_rate_level   logreg         11 folds in    1.6s
step2_rate_level   random_forest  11 folds in  147.9s
step2_rate_level   hist_gb        11 folds in   10.6s
step3_rate_change  logreg         11 folds in    1.2s
step3_rate_change  random_forest  11 folds in  126.3s
step3_rate_change  hist_gb        11 folds in   18.7s
9 models, 99 fold scores


## 6. Scores per fold and in summary

Each fold has its own base rate, so ROC-AUC and PR-AUC are read fold by fold. PR-AUC has to be compared with that fold's base rate, not with a single pooled figure.

In [9]:
STEP_ORDER = list(FEATURE_STEPS)
roc_table = fold_scores.pivot_table(index="fold_id", columns=["family", "step"], values="roc_auc")
roc_table = roc_table[[(fam, st) for fam in FAMILIES for st in STEP_ORDER]]
roc_table.insert(0, "base_rate_pct", (100 * fold_scores.groupby("fold_id")["base_rate"].first()).round(2))
print("ROC-AUC per fold")
roc_table.round(3)

ROC-AUC per fold


family  base_rate_pct   logreg                                     \
step                  baseline step2_rate_level step3_rate_change   
fold_id                                                             
0               11.18    0.801            0.796             0.793   
1               13.62    0.868            0.868             0.869   
2                3.32    0.817            0.814             0.812   
3                3.11    0.861            0.859             0.860   
4                3.29    0.882            0.882             0.889   
5                1.62    0.854            0.854             0.856   
6                3.22    0.874            0.874             0.876   
7                2.62    0.889            0.891             0.892   
8                2.06    0.889            0.890             0.885   
9                1.70    0.785            0.795             0.788   
10               0.59    0.680            0.683             0.690   

family  random_forest                                     hist_gb  \
step         baseline step2_rate_level step3_rate_change baseline   
fold_id                                                             
0               0.812            0.811             0.804    0.811   
1               0.870            0.846             0.858    0.870   
2               0.816            0.828             0.818    0.818   
3               0.864            0.828             0.791    0.868   
4               0.899            0.895             0.885    0.906   
5               0.896            0.895             0.882    0.897   
6               0.892            0.879             0.877    0.889   
7               0.894            0.896             0.846    0.896   
8               0.912            0.909             0.915    0.907   
9               0.830            0.824             0.750    0.834   
10              0.864            0.809             0.880    0.837   

family                                      
step    step2_rate_level step3_rate_change  
fold_id                                     
0                  0.801             0.785  
1                  0.843             0.845  
2                  0.836             0.823  
3                  0.834             0.840  
4                  0.893             0.896  
5                  0.890             0.885  
6                  0.866             0.878  
7                  0.893             0.886  
8                  0.909             0.916  
9                  0.825             0.796  
10                 0.787             0.813

In [10]:
pr_table = fold_scores.pivot_table(index="fold_id", columns=["family", "step"], values="pr_auc")
pr_table = pr_table[[(fam, st) for fam in FAMILIES for st in STEP_ORDER]]
pr_table.insert(0, "base_rate_pct", (100 * fold_scores.groupby("fold_id")["base_rate"].first()).round(2))
print("PR-AUC per fold (compare each with the base rate in the first column)")
pr_table.round(3)

PR-AUC per fold (compare each with the base rate in the first column)


family  base_rate_pct   logreg                                     \
step                  baseline step2_rate_level step3_rate_change   
fold_id                                                             
0               11.18    0.437            0.426             0.416   
1               13.62    0.592            0.593             0.594   
2                3.32    0.277            0.273             0.269   
3                3.11    0.302            0.300             0.303   
4                3.29    0.360            0.359             0.379   
5                1.62    0.201            0.202             0.202   
6                3.22    0.445            0.445             0.449   
7                2.62    0.368            0.370             0.370   
8                2.06    0.365            0.370             0.356   
9                1.70    0.177            0.180             0.176   
10               0.59    0.095            0.099             0.102   

family  random_forest                                     hist_gb  \
step         baseline step2_rate_level step3_rate_change baseline   
fold_id                                                             
0               0.414            0.377             0.371    0.420   
1               0.579            0.494             0.542    0.583   
2               0.277            0.267             0.270    0.299   
3               0.285            0.247             0.278    0.281   
4               0.330            0.317             0.377    0.346   
5               0.186            0.223             0.223    0.187   
6               0.386            0.421             0.436    0.399   
7               0.334            0.360             0.339    0.341   
8               0.365            0.373             0.356    0.339   
9               0.160            0.160             0.139    0.158   
10              0.115            0.132             0.125    0.100   

family                                      
step    step2_rate_level step3_rate_change  
fold_id                                     
0                  0.373             0.359  
1                  0.504             0.515  
2                  0.287             0.290  
3                  0.247             0.227  
4                  0.323             0.328  
5                  0.199             0.198  
6                  0.399             0.416  
7                  0.350             0.351  
8                  0.359             0.350  
9                  0.125             0.154  
10                 0.104             0.097

In [11]:
summary = fold_scores.groupby(["family", "step"]).agg(
    mean_roc_auc=("roc_auc", "mean"), std_roc_auc=("roc_auc", "std"),
    mean_pr_auc=("pr_auc", "mean"), std_pr_auc=("pr_auc", "std")).reset_index()
summary["mean_base_rate_pct"] = round(100 * fold_scores["base_rate"].mean(), 2)
summary = summary.sort_values("mean_roc_auc", ascending=False).reset_index(drop=True)
summary.round(4)

,family,step,mean_roc_auc,std_roc_auc,mean_pr_auc,std_pr_auc,mean_base_rate_pct
0,random_forest,baseline,0.8680,0.0351,0.3120,0.1305,4.21
1,hist_gb,baseline,0.8666,0.0359,0.3139,0.1342,4.21
2,random_forest,step2_rate_level,0.8564,0.0387,0.3066,0.1115,4.21
3,hist_gb,step2_rate_level,0.8525,0.0407,0.2972,0.1206,4.21
4,hist_gb,step3_rate_change,0.8511,0.0438,0.2987,0.1214,4.21
5,random_forest,step3_rate_change,0.8462,0.0496,0.3142,0.1243,4.21
6,logreg,step3_rate_change,0.8372,0.0618,0.3288,0.1387,4.21
7,logreg,step2_rate_level,0.8368,0.0621,0.3288,0.1385,4.21
8,logreg,baseline,0.8364,0.0630,0.3290,0.1399,4.21


**Reading the summary.** Momentum alone gives the best average ROC-AUC in both tree families (random forest 0.8680, histogram gradient boosting 0.8666). Adding the rate level lowers it in both (0.8564 and 0.8525), and adding the rate change and interaction lowers it again (0.8462 and 0.8511). Logistic regression is flat across the three steps (0.8364, 0.8368, 0.8372). Fold 10 (2020) has only 71 declines, and logistic regression scores lowest there (0.680). PR-AUC ranks the families differently from ROC-AUC: logistic regression has the highest mean PR-AUC (0.3290 at baseline, against 0.3120 and 0.3139), so the ranking depends on the metric.

## 7. Does the rate help? A bootstrap that resamples whole quarters

Counties in one quarter share the same national rate and market conditions, so they are not independent. The bootstrap therefore resamples the **44 validation quarters** with replacement (2,000 times). In each draw every fold is scored on the quarters drawn for it, and the fold scores are averaged, the same average used for the summary above. A fold with no declines or no non-declines in a draw is left out of that draw.

The quarter weights are applied inside a tie-aware AUC written with NumPy, which is much faster than calling scikit-learn 200,000 times. The first cell proves that, with equal weights, it gives exactly the scikit-learn value on every fold of every model.

In [12]:
def weighted_auc(gid, yy, w):
    # AUC with row weights; rows are sorted by score, gid numbers the groups of tied scores.
    n_g = gid[-1] + 1
    wp = np.bincount(gid, weights=w * yy, minlength=n_g)
    wn = np.bincount(gid, weights=w * (1.0 - yy), minlength=n_g)
    P, N = wp.sum(), wn.sum()
    if P == 0 or N == 0:
        return np.nan
    neg_before = np.cumsum(wn) - wn
    return float((wp * (neg_before + 0.5 * wn)).sum() / (P * N))


quarter_ids = {q: i for i, q in enumerate(sorted(val["quarter"].unique()))}
val_qidx = val["quarter"].map(quarter_ids).to_numpy()
n_q = len(quarter_ids)

prep = {}
for (family, step), p in all_preds.items():
    for f in fold_ids:
        s, y, qi = p[val_pos[f]], val_y[val_pos[f]], val_qidx[val_pos[f]]
        order = np.argsort(s, kind="mergesort")
        ss = s[order]
        gid = np.cumsum(np.r_[True, ss[1:] != ss[:-1]]) - 1
        prep[(family, step, f)] = (gid, y[order].astype(float), qi[order])

ref = fold_scores.set_index(["family", "step", "fold_id"])["roc_auc"]
max_gap = max(abs(weighted_auc(g, yy, np.ones(len(yy))) - ref[(fam, st, f)])
              for (fam, st, f), (g, yy, _) in prep.items())
print(f"Largest gap between the NumPy AUC and scikit-learn (equal weights): {max_gap:.2e}")
assert max_gap < 1e-9, "The fast AUC does not match scikit-learn."

Largest gap between the NumPy AUC and scikit-learn (equal weights): 2.22e-16


In [13]:
COMPARISONS = [("step2_minus_baseline", "step2_rate_level", "baseline"),
               ("step3_minus_step2", "step3_rate_change", "step2_rate_level"),
               ("step3_minus_baseline", "step3_rate_change", "baseline")]

rng = np.random.default_rng(RANDOM_STATE)
deltas = {(fam, name): [] for fam in FAMILIES for name, _, _ in COMPARISONS}
folds_used = []
t0 = time.time()
for it in range(N_BOOT):
    counts = np.bincount(rng.integers(0, n_q, size=n_q), minlength=n_q).astype(float)
    valid = [f for f in fold_ids
             if (counts[val_qidx[val_pos[f]]] * val_y[val_pos[f]]).sum() > 0
             and (counts[val_qidx[val_pos[f]]] * (1 - val_y[val_pos[f]])).sum() > 0]
    if not valid:
        continue
    folds_used.append(len(valid))
    mean_auc = {}
    for (family, step) in all_preds:
        mean_auc[(family, step)] = np.mean([
            weighted_auc(prep[(family, step, f)][0], prep[(family, step, f)][1],
                         counts[prep[(family, step, f)][2]]) for f in valid])
    for fam in FAMILIES:
        for name, a, b in COMPARISONS:
            deltas[(fam, name)].append(mean_auc[(fam, a)] - mean_auc[(fam, b)])

folds_used = np.array(folds_used)
print(f"{len(folds_used)} of {N_BOOT} draws used, {time.time() - t0:.0f}s | "
      f"all 11 folds scorable in {100 * (folds_used == 11).mean():.1f}% of draws "
      f"(average {folds_used.mean():.2f} folds)")

2000 of 2000 draws used, 58s | all 11 folds scorable in 84.5% of draws (average 10.83 folds)


In [14]:
mean_roc = fold_scores.groupby(["family", "step"])["roc_auc"].mean()


def verdict(point, lo, hi):
    if lo > 0 and point >= MIN_GAIN:
        return "helps"
    if lo > 0:
        return "positive but below +0.01"
    if hi < 0:
        return "hurts"
    return "no significant effect"


boot_rows = []
for fam in FAMILIES:
    for name, a, b in COMPARISONS:
        arr = np.array(deltas[(fam, name)])
        lo, hi = np.percentile(arr, [2.5, 97.5])
        point = float(mean_roc[(fam, a)] - mean_roc[(fam, b)])
        boot_rows.append({"family": fam, "comparison": name, "point_delta": point,
                          "ci_lo": float(lo), "ci_hi": float(hi), "n_boot_used": len(arr),
                          "verdict": verdict(point, lo, hi)})
boot_df = pd.DataFrame(boot_rows)
boot_df.round(4)

,family,comparison,point_delta,ci_lo,ci_hi,n_boot_used,verdict
0,logreg,step2_minus_baseline,0.0005,-0.0008,0.0017,2000,no significant effect
1,logreg,step3_minus_step2,0.0004,-0.0008,0.0020,2000,no significant effect
2,logreg,step3_minus_baseline,0.0008,-0.0006,0.0024,2000,no significant effect
3,random_forest,step2_minus_baseline,-0.0116,-0.0173,-0.0002,2000,hurts
4,random_forest,step3_minus_step2,-0.0101,-0.0193,0.0016,2000,no significant effect
5,random_forest,step3_minus_baseline,-0.0217,-0.0295,-0.0071,2000,hurts
6,hist_gb,step2_minus_baseline,-0.0141,-0.0206,-0.0042,2000,hurts
7,hist_gb,step3_minus_step2,-0.0013,-0.0079,0.0074,2000,no significant effect
8,hist_gb,step3_minus_baseline,-0.0155,-0.0199,-0.0045,2000,hurts


In [15]:
robust_rows = []
for name, _, _ in COMPARISONS:
    sub = boot_df[boot_df["comparison"] == name]
    n_help = int((sub["verdict"] == "helps").sum())
    n_hurt = int((sub["verdict"] == "hurts").sum())
    if n_help >= 2:
        robust = "robust: helps in at least 2 of 3 families"
    elif n_help == 1:
        robust = "not robust: helps in only 1 family"
    elif n_hurt >= 1:
        robust = "no gain; hurts in " + str(n_hurt) + " family/families"
    else:
        robust = "no family shows a gain that clears the rule"
    robust_rows.append({"comparison": name, "families_that_help": n_help,
                        "families_that_hurt": n_hurt, "overall": robust})
pd.DataFrame(robust_rows)

,comparison,families_that_help,families_that_hurt,overall
0,step2_minus_baseline,0,2,no gain; hurts in 2 family/families
1,step3_minus_step2,0,0,no family shows a gain that clears the rule
2,step3_minus_baseline,0,2,no gain; hurts in 2 family/families


**Result.** No step passes the rule (interval excludes zero and a gain of at least +0.01) in any family, so neither adding the rate level nor adding the rate change and interaction is robust. For logistic regression all three changes are tiny (+0.0005, +0.0004, +0.0008) with intervals that include zero, and the upper ends are at most +0.0024, so this data cannot rule out a gain of about +0.002 but nothing near +0.01. For random forest and histogram gradient boosting, adding the rate level lowers ROC-AUC (-0.0116 and -0.0141, with intervals below zero; the random forest interval ends at -0.0002, so it is borderline). Step 3 compared with step 2 shows no significant change in any family.

**Answer on the validation folds.** Once a county's own momentum is known, the mortgage rate adds no measurable predictive power. This rests on the 44 validation quarters. This notebook does not read the Test period, and every choice here was made on the validation folds.

## 8. The recommended model, chosen on validation only

The recommended model is the family and feature step with the highest average ROC-AUC across the 11 folds, with PR-AUC shown next to it. Nothing about the Test period is used. The best family within each step is shown as well, so the recommendation can be read against the question of what the rate adds.

In [16]:
best = summary.iloc[0]
summary["is_recommended"] = summary.index == 0
print(f"Recommended on validation: {best['family']} with {best['step']} "
      f"(mean ROC-AUC {best['mean_roc_auc']:.4f}, mean PR-AUC {best['mean_pr_auc']:.4f}, "
      f"mean base rate {best['mean_base_rate_pct']:.2f}%)")
best_per_step = (summary.sort_values("mean_roc_auc", ascending=False)
                 .groupby("step").head(1).set_index("step").loc[STEP_ORDER]
                 [["family", "mean_roc_auc", "mean_pr_auc"]])
best_per_step.round(4)

Recommended on validation: random_forest with baseline (mean ROC-AUC 0.8680, mean PR-AUC 0.3120, mean base rate 4.21%)


,family,mean_roc_auc,mean_pr_auc
step,,,
baseline,random_forest,0.8680,0.3120
step2_rate_level,random_forest,0.8564,0.3066
step3_rate_change,hist_gb,0.8511,0.2987


**Choice.** Random forest on the baseline (momentum only) has the highest average ROC-AUC (0.8680, PR-AUC 0.3120). Histogram gradient boosting on the baseline is within 0.002 (0.8666), and logistic regression has the higher PR-AUC (0.3290), so the choice between families is not a strong one. The choice is fixed here on validation and is not changed in Phase 6.

## 9. Two diagnostics

**Out-of-range rates.** Tree models cannot predict beyond the values they saw in training. For each fold, this shows the share of validation rows whose rate level or 12-month rate change lies outside the range seen in that fold's training rows.

In [17]:
rows = []
for f in fold_ids:
    tr, va = train_by_fold[f], val[val["fold_id"] == f]
    row = {"fold_id": f}
    for feat in ("rate_level", "rate_roc_12m"):
        lo, hi = tr[feat].min(), tr[feat].max()
        row[feat + "_pct_outside_train_range"] = round(100 * ((va[feat] < lo) | (va[feat] > hi)).mean(), 1)
    rows.append(row)
range_check = pd.DataFrame(rows)
range_check

,fold_id,rate_level_pct_outside_train_range,rate_roc_12m_pct_outside_train_range
0,0,100.0,0.0
1,1,25.3,0.0
2,2,100.0,0.0
3,3,24.7,0.0
4,4,0.0,24.8
5,5,0.0,0.0
6,6,0.0,0.0
7,7,0.0,0.0
8,8,0.0,0.0
9,9,0.0,0.0


**Logistic regression coefficients.** Fitted on all Train rows (never Embargo or Test), on the full six-feature step, with standardized inputs so the coefficients can be compared with each other. This is for reading the direction of each feature. The sign of each coefficient is reported as fitted and not adjusted.

In [18]:
train_all = pd.read_sql(
    "SELECT decline_label, " + ", ".join(ALL_FEATURES) +
    " FROM quarterly_modeling_table WHERE split = 'train'", engine)
print(f"Train rows: {len(train_all):,}")
lr_full = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(
    train_all[ALL_FEATURES].to_numpy(), train_all["decline_label"].to_numpy())
coef = pd.Series(lr_full[-1].coef_[0], index=ALL_FEATURES, name="standardized_coefficient")
coef.round(4).to_frame()

Train rows: 164,460


,standardized_coefficient
momentum_1m,-3.0145
momentum_3m,1.9007
momentum_6m,-1.0919
rate_level,0.2981
rate_roc_12m,-0.0744
rate_momentum_interaction,0.2275


**Reading the diagnostics.** The rate level lies outside the training range for all validation rows in folds 0 and 2, about a quarter in folds 1 and 3, and half in fold 10. The rate change is outside it only in fold 4 (24.8%). This may contribute to the drop in the tree models, because trees cannot predict beyond the range they saw, but it does not explain all of it: folds 0 and 2 are fully outside the range and the random forest barely changes there, while step 3 drops sharply in folds 7 and 9, which are inside it. The three momentum measures overlap, so their coefficients (-3.01, +1.90, -1.09) should be read together. The rate coefficients are small (+0.30, -0.07, +0.23), and a non-zero coefficient inside Train does not mean the feature helps on new data.

## 10. Save the results

The tables go back into Postgres, where Phase 6 reads them. Validation predictions are saved for every model, because Phase 6 needs them to choose decision thresholds on validation data. The only file that goes into git is the baseline lock written in section 4.

In [19]:
val_out = val[["fold_id", "county_fips", "quarter", "label_quarter_end", "decline_label"]].copy()
val_out["quarter"] = val_out["quarter"].dt.date
val_out["label_quarter_end"] = val_out["label_quarter_end"].dt.date
for (family, step), p in all_preds.items():
    val_out[f"proba_{family}_{step}"] = p

fold_scores.to_sql("phase5_fold_scores", engine, if_exists="replace", index=False)
summary.to_sql("phase5_step_summary", engine, if_exists="replace", index=False)
boot_df.to_sql("phase5_bootstrap_deltas", engine, if_exists="replace", index=False)
val_out.to_sql("phase5_validation_predictions", engine, if_exists="replace", index=False)

with engine.connect() as conn:
    for t in ("phase5_fold_scores", "phase5_step_summary", "phase5_bootstrap_deltas",
              "phase5_validation_predictions"):
        print(f"{t:32s} {conn.execute(text(f'SELECT COUNT(*) FROM {t}')).scalar():>8,} rows")

phase5_fold_scores                     99 rows
phase5_step_summary                     9 rows
phase5_bootstrap_deltas                 9 rows
phase5_validation_predictions     112,734 rows


## Status

Phase 5 is complete and verified on the real data: the fold checks pass, all 9 models are scored on all 11 folds, the baseline is locked in `outputs/phase5_baseline_lock.json`, the fast AUC matches scikit-learn, and the four tables are saved (99, 9, 9 and 112,734 rows). Recommended on validation: random forest on the baseline. No rate step helps in any family. The Test period is not read in this notebook. It is scored in Phase 6 with frozen models.